# SAIL 2025 - Crowd Flow Forecasting (Starter - EXERCISES)

**AIM-TT Learning Module - Prediction based on sensor data**

> 📝 **This is the beginner-friendly EXERCISES version.**
>
> You are NOT asked to write whole functions from a blank page. Instead, the
> *conceptually important* lines are replaced with `____` (fill-in-the-blank).
> Every exercise cell has:
> - a short **🎯 Goal** line
> - **💡 Hints** pointing at the exact pandas / sklearn method to use
> - an **Expected shape / output** so you know when your answer is right
>
> If you get stuck, peek at the fully-solved [`01_crowd_forecasting_starter.ipynb`](01_crowd_forecasting_starter.ipynb).
> Cells use `print("TODO ...")` instead of raising errors, so the rest of the
> notebook keeps running even if you skip a step.

---

## Learning Objectives
By the end of this notebook you will be able to:
1. Load and explore the SAIL 2025 visitor-flow sensor dataset
2. Understand the sensor network (GASA / GVCV / CMSA) and directional flow measurements
3. Engineer time-series features (lags, rolling statistics, temporal features)
4. Integrate weather data as an external predictor
5. Compare simple baseline forecasters with a **LightGBM** model and understand *why* gradient-boosted trees are a good fit
6. Extend to **quantile regression** for uncertainty estimation
7. Produce multi-step forecasts (4-hour horizon)

### Target Audience
Anyone starting out with **time-series forecasting**: crowd managers, junior data scientists, students, or technical stakeholders who want a hands-on introduction. No prior forecasting experience is assumed - the notebook walks from naive baselines up to a production-style model.

### Dataset
- **Period**: August 20-24, 2025 (SAIL Amsterdam)
- **Resolution**: 3-minute intervals
- **Sensors**: 74 directional pedestrian-flow counters across Amsterdam

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AiMTT-project/UC7-SAIL/blob/main/3.3%20Crowd%20forecasting/Assignment/01_crowd_forecasting_starter_exercises.ipynb)


In [1]:
# --- Environment setup (safe to run on Colab, Kaggle, or a local venv) ---
# Installs everything this notebook imports. Comment this cell out once your
# environment is ready to avoid repeated installs.
%pip install --quiet \
    "pandas>=2.0" "numpy>=1.26" \
    "matplotlib>=3.8" "seaborn>=0.13" \
    "scikit-learn>=1.3" "lightgbm>=4.0" \
    "meteostat>=1.6" \
    "nbformat>=4.2"

# If running on Google Colab, upload the dataset CSV manually or mount Drive.
# The notebook expects it at: ../Dataset/SAIL2025_LVMA_data_3min_20August-25August2025_flow.csv
# On Colab you can instead place the file at /content/ and set DATA_PATH accordingly.
import sys
print(f"Python: {sys.version.split()[0]}")
try:
    import google.colab  # noqa: F401
    print("Running on Google Colab - remember to upload the dataset CSV.")
except ImportError:
    pass


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.5/506.5 kB 15.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
Python: 3.12.13
Running on Google Colab - remember to upload the dataset CSV.


---
## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

print('Setup complete ✓')

---
## 1. Data Loading & First Look

The dataset was collected during **SAIL Amsterdam 2025** (August 20–24). Pedestrian-flow sensors (LVMA cameras) count the number of people passing in each direction every 3 minutes.

**Sensor naming convention:**
- `GASA-*` - Sensors along the route near Amsterdam Centraal Station area
- `GVCV-*` - Sensors near the **ferry terminals**, measuring inflow and outflow of people boarding/leaving the ferries
- `CMSA-*` - Sensors located in the **city centre** (busy pedestrian streets and squares)

Each sensor has **two azimuths** (e.g., `GVCV-01_40` and `GVCV-01_220`) representing opposite walking directions.


In [ ]:
import os
import urllib.request

# Dataset path configuration (compatible with local environments and Google Colab)
data_filename = "SAIL2025_LVMA_data_3min_20August-25August2025_flow.csv"
data_url = "https://raw.githubusercontent.com/AiMTT-project/UC7-SAIL/main/3.3%20Crowd%20forecasting/SAIL2025_LVMA_data_3min_20August-25August2025_flow.csv"

candidate_paths = [
    data_filename,
    os.path.join("..", data_filename),
    os.path.join("..", "Dataset", data_filename),
    os.path.join("sample_data", data_filename),
    os.path.join("/content/sample_data", data_filename),
    os.path.join("/content", data_filename),
]

DATA_PATH = None
for p in candidate_paths:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    os.makedirs("sample_data", exist_ok=True)
    DATA_PATH = os.path.join("sample_data", data_filename)
    print(f"Downloading {data_filename} from GitHub...")
    urllib.request.urlretrieve(data_url, DATA_PATH)
    print(f"Saved to {DATA_PATH}")

print(f"Using dataset: {DATA_PATH}")
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])


In [ ]:
# Separate sensor columns from metadata columns
META_COLS = ['timestamp', 'hour', 'minute', 'day', 'month', 'weekday', 'is_weekend']
SENSOR_COLS = [c for c in df.columns if c not in META_COLS]

print(f'Number of sensor columns: {len(SENSOR_COLS)}')
print(f'Sensor prefixes: {sorted(set(c.split("-")[0] for c in SENSOR_COLS))}')
print(f'\nFirst 10 sensor columns:\n{SENSOR_COLS[:10]}')

In [ ]:
# Basic statistics for a few representative sensors
sample_sensors = ['GVCV-01_40', 'GVCV-01_220', 'GASA-01-A1_135', 'GASA-01-A1_315',
                  'CMSA-GAWW-11_120', 'CMSA-GAWW-11_300']

df[sample_sensors].describe().round(1)

---
## 2. Exploratory Data Analysis

Before building any model we need to understand the **temporal patterns** in the data:
- What does the daily rhythm look like?
- Are there differences between weekdays and weekends?
- Which sensors see the most traffic?

In [ ]:
# 2.1 - Time series of selected sensors
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

sensor_pairs = [
    ('GVCV-01_40', 'GVCV-01_220', 'GVCV-01 (Ferry Terminal)'),
    ('GASA-01-A1_135', 'GASA-01-A1_315', 'GASA-01-A1 (Central Station)'),
    ('CMSA-GAWW-11_120', 'CMSA-GAWW-11_300', 'CMSA-GAWW-11 (City Centre)'),
]

for ax, (s1, s2, title) in zip(axes, sensor_pairs):
    ax.plot(df['timestamp'], df[s1], label=f'Azimuth {s1.split("_")[1]}°', alpha=0.7)
    ax.plot(df['timestamp'], df[s2], label=f'Azimuth {s2.split("_")[1]}°', alpha=0.7)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Flow count (3 min)')
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.suptitle('Visitor Flow Time Series - SAIL 2025', fontsize=14, fontweight='bold', y=1.02)
plt.show()


In [ ]:
# 2.2 - Average hourly profile (all days)
hourly_avg = df.groupby('hour')[sample_sensors].mean()

fig, ax = plt.subplots(figsize=(12, 5))
hourly_avg.plot(ax=ax, marker='o', linewidth=2)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Average Flow Count (3 min)')
ax.set_title('Average Hourly Profile - Selected Sensors', fontweight='bold')
ax.set_xticks(range(24))
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# 2.3 - Heatmap: sensor activity across the event
# Aggregate to hourly resolution for visibility
df_hourly = df.set_index('timestamp')[SENSOR_COLS].resample('1h').sum()

# Select top 20 busiest sensors
top_sensors = df_hourly.sum().nlargest(20).index.tolist()

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(df_hourly[top_sensors].T, cmap='YlOrRd', ax=ax,
            xticklabels=12,  # show every 12th hour label
            yticklabels=True)
ax.set_title('Hourly Visitor Flow Heatmap - Top 20 Sensors', fontweight='bold')
ax.set_xlabel('Time')
ax.set_ylabel('Sensor')
plt.tight_layout()
plt.show()

In [ ]:
# 2.4 - Weekend vs weekday comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, (weekend_val, label) in zip(axes, [(0, 'Weekdays'), (1, 'Weekend')]):
    subset = df[df['is_weekend'] == weekend_val]
    hourly = subset.groupby('hour')[sample_sensors[:2]].mean()
    hourly.plot(ax=ax, marker='o', linewidth=2)
    ax.set_title(f'{label}', fontweight='bold')
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Avg Flow Count')
    ax.set_xticks(range(24))
    ax.legend(fontsize=8)

plt.suptitle('Weekday vs Weekend Profiles - GVCV-01', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Weather Data Integration

SAIL is an outdoor event - crowd volumes are strongly influenced by weather. We use the **meteostat** library to fetch historical weather for Amsterdam and merge it with the sensor data.

**Why weather matters:**
- Rain → fewer visitors
- High temperature → more visitors but potential heat-related safety concerns
- Wind → may affect waterfront areas differently

In [ ]:
from datetime import datetime
import numpy as np

# --- Fetch weather from meteostat (Amsterdam Central) ---
# The meteostat v2.x API uses lowercase: hourly(), not Hourly()
try:
    from meteostat import Point, hourly as meteostat_hourly
    amsterdam = Point(52.3807, 4.8968, 4)  # lat, lon, altitude (m)
    start, end = datetime(2025, 8, 20), datetime(2025, 8, 26)
    ts = meteostat_hourly(amsterdam, start, end)
    weather = ts.fetch() if not ts.empty else None
except Exception as e:
    print(f'meteostat fetch failed: {e}')
    weather = None

if weather is not None and not weather.empty:
    weather = weather[['temp', 'prcp', 'wspd', 'rhum']].rename(columns={
        'temp': 'temperature_c',
        'prcp': 'precipitation_mm',
        'wspd': 'wind_speed_kmh',
        'rhum': 'relative_humidity'
    })
    print(f'✓ Fetched real weather data: {weather.shape}')
else:
    # Generate realistic synthetic weather for Amsterdam in August
    # Typical: 17-23°C, occasional rain, moderate wind, ~70% humidity
    print('⚠ meteostat data not available - generating synthetic Amsterdam August weather')
    hours = pd.date_range('2025-08-20', '2025-08-24 23:00', freq='h')
    rng = np.random.default_rng(42)
    weather = pd.DataFrame({
        'temperature_c': 18 + 4 * np.sin(np.pi * (hours.hour - 6) / 12) + rng.normal(0, 1.5, len(hours)),
        'precipitation_mm': np.where(rng.random(len(hours)) > 0.85,
                                      rng.exponential(1.5, len(hours)), 0.0),
        'wind_speed_kmh': np.clip(12 + rng.normal(0, 4, len(hours)), 0, None),
        'relative_humidity': np.clip(70 + 10 * np.cos(np.pi * (hours.hour - 14) / 12) + rng.normal(0, 5, len(hours)), 30, 100),
    }, index=hours)
    print(f'✓ Generated synthetic weather: {weather.shape}')

print(f'\nWeather data shape: {weather.shape}')
weather.head(10)

In [ ]:
# Visualize weather alongside a busy sensor
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

# Sensor flow (hourly aggregate for readability)
busy_sensor = 'GVCV-01_40'  # Use a GVCV sensor available in all configs
sensor_hourly = df.set_index('timestamp')[busy_sensor].resample('1h').sum()
axes[0].plot(sensor_hourly.index, sensor_hourly.values, color='steelblue')
axes[0].set_ylabel('Flow (hourly)')
axes[0].set_title(f'{busy_sensor} Flow vs Weather', fontweight='bold')

# Temperature
axes[1].plot(weather.index, weather['temperature_c'], color='orangered')
axes[1].set_ylabel('Temp (°C)')

# Precipitation
axes[2].bar(weather.index, weather['precipitation_mm'].fillna(0), color='royalblue', width=0.04)
axes[2].set_ylabel('Precip (mm)')
axes[2].set_xlabel('Date')

plt.tight_layout()
plt.show()

In [ ]:
# Merge weather into the main dataframe
# Weather is hourly → forward-fill to 3-minute resolution
# Align timestamp timezone with naive hourly index for clean merge
if df['timestamp'].dt.tz is not None:
    df['hour_floor'] = df['timestamp'].dt.tz_convert('Europe/Amsterdam').dt.tz_localize(None).dt.floor('h')
else:
    df['hour_floor'] = df['timestamp'].dt.floor('h')

# Align weather timezone with data (naive local time)
if weather.index.tz is not None:
    weather.index = weather.index.tz_convert('Europe/Amsterdam').tz_localize(None)

weather_cols = ['temperature_c', 'precipitation_mm', 'wind_speed_kmh', 'relative_humidity']

df = df.merge(weather[weather_cols], left_on='hour_floor', right_index=True, how='left')
# Forward-fill for most rows; back-fill any leading NaN
df[weather_cols] = df[weather_cols].ffill().bfill()

# Clean up helper column
df = df.drop(columns=['hour_floor'])

print(f'Weather features added ✓ (NaN remaining: {df[weather_cols].isna().sum().sum()})')
df[['timestamp'] + weather_cols].head()

---
## 4. Feature Engineering

Time-series forecasting requires **features that capture recent history**. We create:

### 4.1 Lag Features
The value at $t-1, t-2, \ldots, t-k$ ("what happened recently?"). We use **40 lags** (= 2 hours of 3-minute data).

### 4.2 Rolling Statistics
Smoothed views over longer windows:
- **30 min** (10 steps): short-term trend
- **1 hour** (20 steps): medium-term trend
- **2 hours** (40 steps): longer trend
- **4 hours** (80 steps): event-scale trend

### 4.3 Temporal Features
Hour and minute of day - crowds have strong daily rhythms.

### 4.4 Weather Features
Temperature, precipitation, wind speed, humidity.


In [ ]:
# --- Feature engineering (provided - study this!) ---
# Beginners: read this function carefully rather than rewriting it from scratch.
# It builds the feature matrix that the model will learn from.
#
# READING EXERCISE 🎯
#   1. Find the line that creates LAG features. How many lags are built?
#      (Answer: range(1, 41) → 40 lags, i.e. the last 2 hours.)
#   2. Find the line that creates ROLLING statistics. Which windows are used?
#   3. Why do we call .fillna(0) at the end?
#      (Hint: the first few rows have NaN because lag40 needs 40 past rows.)

def create_features(df: pd.DataFrame, sensor: str) -> pd.DataFrame:
    """
    Build the feature matrix for a single sensor.
    """
    # Temporal + weather context columns
    context_cols = ['hour', 'minute']
    weather_feature_cols = ['temperature_c', 'precipitation_mm', 'wind_speed_kmh', 'relative_humidity']
    available_weather = [c for c in weather_feature_cols if c in df.columns]

    out = df[['timestamp'] + context_cols + available_weather + [sensor]].copy()

    # Lag features (40 steps = 2 hours)
    for lag in range(1, 41):
        out[f'{sensor}_lag{lag}'] = df[sensor].shift(lag)

    # Rolling statistics
    rolling_windows = {'30min': 10, '1h': 20, '2h': 40, '4h': 80}
    s = df[sensor]
    for label, window_size in rolling_windows.items():
        roll = s.rolling(window_size)
        out[f'{sensor}_mean_{label}'] = roll.mean()
        out[f'{sensor}_std_{label}'] = roll.std()

    return out.fillna(0)


# Demonstrate on one sensor
TARGET_SENSOR = 'GVCV-01_40'
features_df = create_features(df, TARGET_SENSOR)

feature_cols = [c for c in features_df.columns if c not in ['timestamp', TARGET_SENSOR]]
print(f'Number of features: {len(feature_cols)}')
print(f'Feature groups:')
print(f'  - Lag features:     {sum(1 for c in feature_cols if "lag" in c)}')
print(f'  - Rolling features: {sum(1 for c in feature_cols if "mean" in c or "std" in c)}')
print(f'  - Temporal:         {sum(1 for c in feature_cols if c in ["hour", "minute"])}')
print(f'  - Weather:          {sum(1 for c in feature_cols if c in ["temperature_c", "precipitation_mm", "wind_speed_kmh", "relative_humidity"])}')
features_df.head()


---
## 5. Train/Test Split

**Critical for time series**: we split **temporally**, not randomly.

- **Train**: August 20–22 (first 3 days)
- **Test**: August 23–24 (last 2 days - the weekend)

This simulates reality: we train on historical data and predict the future.

In [ ]:
# Temporal train/test split
SPLIT_DATE = '2025-08-23'

train_mask = features_df['timestamp'] < SPLIT_DATE
test_mask = features_df['timestamp'] >= SPLIT_DATE

# Target: the NEXT 3-minute flow count (1-step ahead)
# At time t, features capture the state up to t (lags from t-1, rolling stats up to t).
# The target y_{t+1} is strictly in the future → no data leakage.
y_target = df[TARGET_SENSOR].shift(-1)

# Drop last row (no target available)
valid_train = train_mask & y_target.notna()
valid_test = test_mask & y_target.notna()

X_train = features_df.loc[valid_train, feature_cols]
y_train = y_target.loc[valid_train]

X_test = features_df.loc[valid_test, feature_cols]
y_test = y_target.loc[valid_test]
test_timestamps = features_df.loc[valid_test, 'timestamp']

print(f'Train: {X_train.shape[0]:,} samples ({features_df.loc[valid_train, "timestamp"].min().date()} → {features_df.loc[valid_train, "timestamp"].max().date()})')
print(f'Test:  {X_test.shape[0]:,} samples ({features_df.loc[valid_test, "timestamp"].min().date()} → {features_df.loc[valid_test, "timestamp"].max().date()})')
print(f'Target: y_{{t+1}} - predicting the next 3-minute interval')

---
## 6. Forecasting Models - From Baselines to LightGBM

Before reaching for a fancy model, **always start with simple baselines**. They tell you whether the problem is hard at all and give you a yardstick to measure improvement against.

We'll compare **four** models on the exact same 1-step-ahead test set.

### 6.1 Naive Persistence - *"the next minute looks like this minute"*
$$\hat{y}_{t+1} = y_t$$
Just repeat the last observed value. Zero training, zero parameters.

### 6.2 Seasonal Naive - *"this morning will look like yesterday morning"*
$$\hat{y}_{t+1} = y_{t+1 - S} \quad \text{with } S = 24\text{ h} = 480 \text{ steps}$$
Captures the daily rhythm without any training.

### 6.3 Linear Regression
$$\hat{y}_{t+1} = w_0 + w_1 \cdot \text{lag}_1 + w_2 \cdot \text{lag}_2 + \ldots$$
Fast and interpretable, but only models **straight-line** relationships.

### 6.4 LightGBM - Gradient-Boosted Decision Trees
LightGBM grows hundreds of small **decision trees**. Each new tree is trained to **fix the mistakes** of the previous trees:

```
Tree 1:   rough guess based on lag_1, hour, ...
Tree 2:   corrects Tree 1's errors
...
Tree 300: tiny remaining adjustment

Final prediction = sum of all tree outputs
```

Strengths for this problem:
- Captures non-linear patterns (rush hours, weekend ↔ weekday flips, weather thresholds)
- Handles many features without much tuning
- Supports **quantile loss** (needed for the prediction intervals in Section 7)

The cell below trains all four. The two naive baselines and the Linear-Regression block are **provided so you can read and run them**. Your fill-in is only the LightGBM call.


In [ ]:
import lightgbm as lgb
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Test indices we'll evaluate on (same for every model)
test_idx = X_test.index

# --- Baseline 1: Naive persistence (PROVIDED) ---
naive_pred = df[TARGET_SENSOR].loc[test_idx].values

# --- Baseline 2: Seasonal naive - same minute yesterday (PROVIDED) ---
SEASON = 480  # 24h × 60min / 3min
seasonal_pred = df[TARGET_SENSOR].shift(SEASON - 1).loc[test_idx].values
seasonal_pred = np.nan_to_num(seasonal_pred, nan=0.0)

# --- Model 3: Linear Regression (PROVIDED) ---
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = np.maximum(0, lr.predict(X_test))

# --- Model 4: LightGBM (YOUR TURN) ---
# 🎯 Goal: train a LightGBM regressor on (X_train, y_train) and predict on X_test.
# 💡 Hints:
#   - lgb.LGBMRegressor(objective='regression', n_estimators=300, learning_rate=0.05,
#                       max_depth=6, num_leaves=31, random_state=42, verbosity=-1)
#   - .fit(X_train, y_train)
#   - .predict(X_test)  →  clip to >= 0 with np.maximum(0, ...)
# Expected: y_pred is a numpy array of length len(X_test); model_median is a fitted LGBMRegressor.
try:
    model_median = lgb.LGBMRegressor(
        objective='regression',
        n_estimators=300,
        learning_rate=____,   # FILL IN: 0.05
        max_depth=6,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=-1,
    )
    model_median.____(X_train, y_train)               # FILL IN: fit
    y_pred = np.maximum(0, model_median.____(X_test)) # FILL IN: predict
except (TypeError, AttributeError, NameError):
    print('TODO: train LightGBM - falling back to Linear Regression so the rest of the notebook still runs.')
    model_median = lr
    y_pred = lr_pred

# --- Compare ---
def score(name, y_true, y_pred):
    return {
        'Model': name,
        'MAE':  mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
    }

baseline_results = pd.DataFrame([
    score('Naive Persistence',    y_test.values, naive_pred),
    score('Seasonal Naive (24h)', y_test.values, seasonal_pred),
    score('Linear Regression',    y_test.values, lr_pred),
    score('LightGBM',             y_test.values, y_pred),
])
print(baseline_results.to_string(index=False, float_format='%.2f'))

fig, ax = plt.subplots(figsize=(9, 4))
baseline_results.set_index('Model')[['MAE', 'RMSE']].plot.barh(
    ax=ax, color=['steelblue', 'orange']
)
ax.set_xlabel('Error (people / 3 min)')
ax.set_title(f'{TARGET_SENSOR} - 1-step-ahead forecast: model comparison', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('\n💬 READING EXERCISE: Which baseline is the toughest to beat? '
      'Does LightGBM beat Linear Regression by a small or a large margin?')


In [ ]:
# Visualize predictions vs actuals
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(test_timestamps.values, y_test.values, label='Actual', alpha=0.7, linewidth=1)
ax.plot(test_timestamps.values, y_pred, label='Predicted', alpha=0.7, linewidth=1)
ax.set_xlabel('Time')
ax.set_ylabel('Flow Count')
ax.set_title(f'{TARGET_SENSOR} - Predicted vs Actual (1-step ahead)', fontweight='bold')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d %H:%M'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance - which features matter most?
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_median.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
importance.head(20).plot.barh(x='feature', y='importance', ax=ax, legend=False)
ax.set_title('Top 20 Feature Importances', fontweight='bold')
ax.set_xlabel('Importance (split count)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## 7. Quantile Regression - Uncertainty Estimation

### Why Uncertainty Matters for Crowd Management
A point prediction tells you "we expect 50 people" - but that's not enough for safety-critical decisions. What crowd managers really need is:

> *"We are 90% confident that between 30 and 80 people will pass this point in the next 3 minutes."*

### What is Quantile Regression?
Instead of minimizing the mean squared error (which gives the conditional mean), **quantile regression** minimizes the **pinball loss** to estimate specific percentiles:

$$L_\tau(y, \hat{y}) = \begin{cases} \tau \cdot (y - \hat{y}) & \text{if } y \geq \hat{y} \\ (1-\tau) \cdot (\hat{y} - y) & \text{if } y < \hat{y} \end{cases}$$

where $\tau$ is the target quantile (e.g., 0.05 for the lower 5%, 0.95 for the upper 95%).

**We train 3 models:**
- **Lower (τ = 0.05)**: 5th percentile → lower bound
- **Median (τ = 0.50)**: 50th percentile → central estimate
- **Upper (τ = 0.95)**: 95th percentile → upper bound

Together, the lower and upper bounds form a **90% prediction interval**.

### How Do We Score an Interval?
Point metrics (MAE, RMSE) only judge the median. For the interval itself we add two complementary metrics:

| Metric | Formula | What it tells us |
|--------|---------|------------------|
| **MIC** - Mean Interval Coverage | $\dfrac{1}{N}\sum_t \mathbb{1}\!\left[L_t \le y_t \le U_t\right]$ | Fraction of actuals inside the interval. Should be ≈ 0.90 for a 90 % PI. |
| **MIW** - Mean Interval Width | $\dfrac{1}{N}\sum_t (U_t - L_t)$ | Average width of the interval (smaller = sharper, but only useful if MIC stays calibrated). |

A good probabilistic forecaster wants **MIC close to the nominal level** *and* **MIW as small as possible**.


In [ ]:
# 🎯 GOAL: Train THREE quantile models (lower / median / upper) to get a
#          90% prediction interval instead of a single number.
#
# 💡 HINTS
#   - For quantile regression, pass objective='quantile' and alpha=<q>
#     where q is the target quantile (0.05, 0.50, 0.95).
#   - Everything else is the same as the point model above.
#   - After predicting, we must enforce:
#       lower ≥ 0          (no negative counts)
#       median ≥ lower     (quantile ordering)
#       upper ≥ median     (quantile ordering)
#
# Expected output: three fitted models, and three prediction arrays of length len(X_test).
# You should see roughly 85–90% of y_test values fall between pred_lower and pred_upper.

quantiles = {'lower': 0.05, 'median': 0.50, 'upper': 0.95}
models = {}

for name, alpha in quantiles.items():
    print(f'Training {name} model (quantile={alpha})...')
    # ▼▼▼ FILL IN ▼▼▼
    # model = lgb.LGBMRegressor(
    #     objective=____,        # 'quantile'
    #     alpha=____,            # the q value for this iteration
    #     n_estimators=300,
    #     learning_rate=0.05,
    #     max_depth=6,
    #     verbosity=-1,
    #     random_state=42,
    # )
    # model.fit(X_train, y_train)
    # models[name] = model
    # ▲▲▲ FILL IN ▲▲▲
    pass  # remove this once you fill in above

# Predict the three quantiles (works once `models` is populated)
if set(models.keys()) == {'lower', 'median', 'upper'}:
    pred_lower = models['lower'].predict(X_test)
    pred_median = models['median'].predict(X_test)
    pred_upper = models['upper'].predict(X_test)

    # 💡 Enforce non-negativity and ordering (already written for you):
    pred_lower = np.maximum(0, pred_lower)
    pred_median = np.maximum(pred_lower, pred_median)
    pred_upper = np.maximum(pred_median, pred_upper)
    print('Quantile models trained ✓')
else:
    print("TODO: train the 3 quantile models above - using placeholders for now.")
    pred_lower = np.zeros(len(X_test))
    pred_median = np.zeros(len(X_test))
    pred_upper = np.zeros(len(X_test))



In [ ]:
# Visualize prediction intervals + score them with MAE / RMSE / MIC / MIW
fig, ax = plt.subplots(figsize=(14, 6))

ts = test_timestamps.values

ax.fill_between(ts, pred_lower, pred_upper, alpha=0.3, color='steelblue', label='90% Prediction Interval')
ax.plot(ts, y_test.values, label='Actual', color='black', linewidth=1, alpha=0.8)
ax.plot(ts, pred_median, label='Median Prediction', color='steelblue', linewidth=1.5)

ax.set_xlabel('Time')
ax.set_ylabel('Flow Count (3 min)')
ax.set_title(f'{TARGET_SENSOR} - Quantile Regression with 90% Prediction Interval', fontweight='bold')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d %H:%M'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- Probabilistic-forecast scorecard ----------------------------------------
NOMINAL = 0.90  # we trained for a 90% prediction interval

mae_med = mean_absolute_error(y_test.values, pred_median)
rmse_med = np.sqrt(mean_squared_error(y_test.values, pred_median))
mic = np.mean((y_test.values >= pred_lower) & (y_test.values <= pred_upper))   # Mean Interval Coverage
miw = np.mean(pred_upper - pred_lower)                                          # Mean Interval Width

print('\n- Quantile-forecast scorecard (test set) -')
print(f'  MAE  (median)        : {mae_med:.3f}')
print(f'  RMSE (median)        : {rmse_med:.3f}')
print(f'  MIC  (coverage)      : {mic:.1%}    (target: {NOMINAL:.0%})')
print(f'  MIW  (interval width): {miw:.2f}')

if mic < NOMINAL - 0.05:
    print(f'\nNote: MIC below {NOMINAL:.0%} is common when:')
    print(f'  - Training data is limited ({X_train.shape[0]} samples from ~3 days)')
    print(f'  - Test distribution differs from training (weekend vs weekdays)')
    print(f'  - Data is zero-inflated (many nighttime zeros)')
    print(f'  The Kalman filter in the advanced notebook helps address this.')


---
## 8. Multi-Step Forecasting (4-Hour Horizon)

In the production system, we don't just predict the next 3 minutes - we predict **4 hours ahead** (80 steps × 3 minutes).

### Approach: Direct Multi-Output
Instead of recursively feeding predictions back into the model, the production system uses **direct multi-output forecasting**:
- Train one model that outputs 80 future values simultaneously
- Each step $t+1, t+2, \ldots, t+80$ has its own output

**Advantage**: No error accumulation from recursive predictions.  
**Trade-off**: Requires more training data and computational resources.

> For this starter notebook, we demonstrate the concept with a simplified version.

In [ ]:
HORIZON = 80  # 80 steps × 3 min = 4 hours

# 🎯 GOAL: Build the multi-step target matrix `y_multi`.
#    Column 'y_t+k' contains df[TARGET_SENSOR] shifted k steps INTO THE FUTURE.
#
# 💡 HINTS
#   - "k steps into the future" → df[col].shift(-k)  (NEGATIVE shift).
#   - Use pd.concat([...], axis=1) to stack the columns side-by-side.
#   - Use .rename(f'y_t+{step}') to name each column.
#
# Expected: y_multi.shape == (len(df), 80)  and columns == ['y_t+1', ..., 'y_t+80'].

# ▼▼▼ FILL IN the shift amount: replace ____ with -step ▼▼▼
y_multi = pd.concat(
    [df[TARGET_SENSOR].shift(____).rename(f'y_t+{step}') for step in range(1, HORIZON + 1)],
    axis=1,
)
# ▲▲▲ FILL IN ▲▲▲

# Align features and targets, drop rows where ANY future target is NaN
valid_mask = y_multi.notna().all(axis=1) & train_mask
X_train_multi = features_df.loc[valid_mask, feature_cols]
y_train_multi = y_multi.loc[valid_mask]

print(f'Training samples for multi-step: {X_train_multi.shape[0]:,}')
print(f'Target shape: {y_train_multi.shape} (samples × horizon steps)')
assert y_train_multi.shape[1] == HORIZON, "y_multi should have 80 columns - check your shift value."


In [ ]:
from sklearn.multioutput import MultiOutputRegressor

# 🎯 GOAL: Train one LightGBM regressor per future step in DEMO_STEPS.
#          Store them in `multi_models[step]`.
#
# 💡 HINTS
#   - The target column for step `step` is named f'y_t+{step}' inside y_train_multi.
#   - Reuse lgb.LGBMRegressor(objective='regression', n_estimators=200,
#     learning_rate=0.05, max_depth=6, verbosity=-1, random_state=42).
#   - Fit with model.fit(X_train_multi, y_train_multi[<column name>]).

DEMO_STEPS = [1, 5, 10, 20, 40, 80]
multi_models = {}

for step in DEMO_STEPS:
    col = f'y_t+{step}'
    model = lgb.LGBMRegressor(
        objective='regression',
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        verbosity=-1,
        random_state=42,
    )
    # ▼▼▼ FILL IN ▼▼▼
    # model.fit(____, y_train_multi[____])
    # multi_models[step] = ____
    # ▲▲▲ FILL IN ▲▲▲
    pass  # remove this once you fill in above

if len(multi_models) == len(DEMO_STEPS):
    print(f'Trained models for steps: {DEMO_STEPS}')
    print(f'That corresponds to: {[s*3 for s in DEMO_STEPS]} minutes ahead')
else:
    print(f"TODO: fit models for each step. Trained so far: {list(multi_models.keys())}")


In [ ]:
# Demonstrate a 4-hour forecast from a single point in time
# Pick a point in the test set (noon on Aug 24 - busy period)
forecast_origin_idx = features_df[
    (features_df['timestamp'] >= '2025-08-24 12:00') &
    (features_df['timestamp'] < '2025-08-24 12:03')
].index[0]

X_origin = features_df.loc[[forecast_origin_idx], feature_cols]
origin_time = features_df.loc[forecast_origin_idx, 'timestamp']

# Generate forecasts for selected steps
forecast_times = [origin_time + pd.Timedelta(minutes=step*3) for step in DEMO_STEPS]
forecast_values = [multi_models[step].predict(X_origin)[0] for step in DEMO_STEPS]

# Also get the actual future values for comparison
actual_future = df.set_index('timestamp')[TARGET_SENSOR].loc[
    origin_time:origin_time + pd.Timedelta(hours=4)
]

fig, ax = plt.subplots(figsize=(14, 5))

# Historical (last 2 hours)
hist_start = origin_time - pd.Timedelta(hours=2)
historical = df.set_index('timestamp')[TARGET_SENSOR].loc[hist_start:origin_time]
ax.plot(historical.index, historical.values, color='gray', linewidth=1.5, label='Historical')

# Actual future
ax.plot(actual_future.index, actual_future.values, color='black', linewidth=1.5, label='Actual')

# Forecast points
ax.scatter(forecast_times, forecast_values, color='red', s=80, zorder=5, label='Forecast')
ax.plot(forecast_times, forecast_values, color='red', linewidth=1, linestyle='--', alpha=0.5)

ax.axvline(origin_time, color='green', linestyle='--', alpha=0.7, label='Forecast origin')
ax.set_xlabel('Time')
ax.set_ylabel('Flow Count')
ax.set_title(f'{TARGET_SENSOR} - 4-Hour Forecast from {origin_time.strftime("%b %d %H:%M")}', fontweight='bold')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.tight_layout()
plt.show()

---
## 9. Evaluation Metrics Summary

Let's evaluate performance across different forecast horizons.

In [ ]:
# Evaluate at each forecast horizon
results = []

for step in DEMO_STEPS:
    col = f'y_t+{step}'
    # Get valid test indices
    valid_test = test_mask & y_multi[col].notna()
    if valid_test.sum() == 0:
        continue

    y_true = y_multi.loc[valid_test, col]
    y_hat = multi_models[step].predict(features_df.loc[valid_test, feature_cols])

    results.append({
        'Horizon': f't+{step}',
        'Minutes Ahead': step * 3,
        'MAE': mean_absolute_error(y_true, y_hat),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_hat)),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False, float_format='%.2f'))

# Plot MAE vs horizon
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(results_df['Minutes Ahead'], results_df['MAE'], marker='o', linewidth=2)
ax.set_xlabel('Forecast Horizon (minutes)')
ax.set_ylabel('MAE (people / 3 min)')
ax.set_title('Forecast Error vs Horizon', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Key Takeaways

1. **Always benchmark against simple baselines.** Naive persistence and seasonal naive set the bar - if your fancy model can't beat them, something is wrong.
2. **LightGBM wins on this problem** because it captures non-linear, interaction-heavy patterns that linear regression cannot.
3. **Per-sensor models scale better** than one global model when sensors have different traffic patterns.
4. **Multi-output prediction** lets a single model produce all 80 forecast steps in one inference call - no error accumulation.
5. **Quantile regression provides uncertainty.** The 90% prediction interval - scored with **MIC** (coverage) and **MIW** (width) - captures the range of plausible outcomes, which is critical for crowd-management decisions.\n
6. **Accuracy degrades with horizon.** 3-minute-ahead predictions are much more accurate than 4-hour predictions. This is a fundamental trade-off.
7. **Weather adds context.** Temperature and precipitation correlate with visitor volumes at outdoor events.

### Going Further
The advanced notebook builds on this with:
- **All 48 sensors** trained simultaneously
- **Kalman filter** for online bias correction (adapting to real-time conditions)
- **SHAP analysis** for feature attribution
- **Interactive Plotly dashboards** for exploring forecasts

→ Continue to **Notebook 02** for the advanced deep-dive.


---
## 11. Exercises for Practice

Try these to deepen your understanding:

1. **Different sensors**: Modify `TARGET_SENSOR` and re-run the pipeline. Which sensors are easier to predict?
2. **Feature ablation**: Remove rolling statistics or weather features and observe MAE change.
3. **Hyperparameter tuning**: Try different `learning_rate`, `max_depth`, `n_estimators`. Use cross-validation.
4. **Multi-sensor model**: Train a single model that uses ALL sensors as features (cross-sensor information).
5. **Calibration analysis**: For the 90% prediction interval, check whether **MIC ≈ 90%** *and* whether **MIW** is acceptably small. If MIC is too low → the model is over-confident; if MIW is huge → the interval is sharp on paper but useless in practice.
